## Settings

In [1]:
import sys
import os
from pathlib import Path
from tqdm.notebook import tqdm

In [25]:
# ===== Environment detection =====
IS_KAGGLE = "KAGGLE_URL_BASE" in os.environ

print("Running on Kaggle:", IS_KAGGLE)


# ===== Path settings =====
if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/stanford-rna-3d-folding-2")
    SRC_DIR = Path("/kaggle/input/rna2-source-codes")
    WORK_DIR = Path("/kaggle/working")
else:
    BASE_DIR = Path.cwd().parent
    DATA_DIR = BASE_DIR / "data" / "stanford-rna-3d-folding-2"
    SRC_DIR = BASE_DIR / "src"
    SCRIPT_DIR = BASE_DIR / "scripts"
    WORK_DIR = BASE_DIR / "data" / "output"


print("DATA_DIR:", DATA_DIR)
print("SRC_DIR:", SRC_DIR)
print("WORK_DIR:", WORK_DIR)

Running on Kaggle: False
DATA_DIR: /Users/tatsuki/work/kaggle/kauto/competitions/rna2/data/stanford-rna-3d-folding-2
SRC_DIR: /Users/tatsuki/work/kaggle/kauto/competitions/rna2/src
WORK_DIR: /Users/tatsuki/work/kaggle/kauto/competitions/rna2/data/output


In [3]:
if IS_KAGGLE:
    from IPython import get_ipython
    get_ipython().system("pip install --no-deps --find-links=/kaggle/input/biopython/biopython-1.85-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl biopython")
else:
    print("Skipping Kaggle-only pip install (not running on Kaggle)")

Skipping Kaggle-only pip install (not running on Kaggle)


In [ ]:
# import Bio
# from Bio.Align import PairwiseAligner

# print("Biopython version:", Bio.__version__)
# print("PairwiseAligner is available")

Biopython version: 1.86
PairwiseAligner is available


In [9]:
# ===== File paths =====
train_seq_path = DATA_DIR / "train_sequences.csv"
train_label_path = DATA_DIR / "train_labels.csv"
valid_seq_path = DATA_DIR / "validation_sequences.csv"
valid_label_path = DATA_DIR / "validation_labels.csv"
test_seq_path = DATA_DIR / "test_sequences.csv"

print("Train seq:", train_seq_path.exists())
print("Train label:", train_label_path.exists())
print("Val seq:", valid_seq_path.exists())
print("Test seq:", test_seq_path.exists())

Train seq: True
Train label: True
Val seq: True
Test seq: True


In [10]:
# ===== Imports =====
sys.path.append(str(SRC_DIR))
from baseline.data import load_sequences, load_labels
from baseline.template_model import TemplateRepository
from baseline.search import seq_identity
from baseline.predict import generate_submission

if not IS_KAGGLE:
    sys.path.append(str(SCRIPT_DIR))
    from kabsch_utils import align_and_rmsd
    from backbone_utils import extract_C1p_coords, extract_coords_from_submission

In [11]:
if IS_KAGGLE:
    import runpy

    module_globals = runpy.run_path("/kaggle/usr/lib/tm-score-permutechains/metric.py")
    score = module_globals['score']

## Load Data

In [12]:
# ===== Load data =====
train_seq = load_sequences(train_seq_path)
train_labels = load_labels(train_label_path)

valid_seq = load_sequences(valid_seq_path)
valid_labels = load_labels(valid_label_path)

test_seq = load_sequences(test_seq_path)

print("Train size:", len(train_seq))
print("Val size:", len(valid_seq))
print("Test size:", len(test_seq))

Train size: 5716
Val size: 28
Test size: 28


## Create template repository

In [13]:
# Fit repository and generate a submission
repo = TemplateRepository()

repo.fit(train_seq, train_labels)
print('templates count:', len(repo.templates))

templates count: 5716


## Predict on validation set

In [14]:
# NOTE: 並列化の意味は小さいかも
valid_pred = generate_submission(valid_seq, repo, seq_identity, n_structures=5, n_jobs=4)
print('submission rows:', len(valid_pred))

valid_pred.head()

submission rows: 9762


,ID,resname,resid,x_1,y_1,z_1,x_2,y_2,z_2,x_3,y_3,z_3,x_4,y_4,z_4,x_5,y_5,z_5
0,8ZNQ_1,A,1,12.133,-11.862,1.757,166.708,162.080,161.200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8ZNQ_2,C,2,NaN,NaN,NaN,167.160,157.024,159.502,193.176,143.370,155.162,NaN,NaN,NaN,9.981,-13.553,-4.518
2,8ZNQ_3,C,3,NaN,NaN,NaN,164.384,147.392,163.683,188.045,141.202,155.911,NaN,NaN,NaN,8.575,-8.310,5.003
3,8ZNQ_4,G,4,11.260,-7.167,5.547,160.163,145.020,166.617,NaN,NaN,NaN,-15.397,-2.768,4.068,12.275,-5.392,7.117
4,8ZNQ_5,U,5,12.239,-0.828,5.049,155.045,145.136,168.216,183.603,138.622,153.965,-6.594,-8.409,3.503,17.668,-2.604,8.109


## Evaluate on validation set

In [15]:
#　prepare groud truth
sol = valid_labels.copy()
sub = valid_pred.copy()

# extract target_id
sol['target_id'] = sol['ID'].apply(lambda x: '_'.join(str(x).split('_')[:-1]))
sub['target_id'] = sub['ID'].apply(lambda x: '_'.join(str(x).split('_')[:-1]))

In [20]:
if not IS_KAGGLE:
    # Evaluation helper: imported from external module
    from evaluate import evaluate_submission

    df, summary = evaluate_submission(pred_df=valid_pred, true_df=valid_labels)

    print('Validation summary:')
    print(summary)

Validation summary:
{'n_targets': 28, 'n_evaluated': 28, 'mean_rmsd': 36.195305345602634, 'median_rmsd': 30.48877698800794}


In [21]:
if IS_KAGGLE:
    results = []
    for target_id, group_native in tqdm(sol.groupby('target_id')):
        group_predicted = sub[sub['target_id'] == target_id]

        result = score(group_native, group_predicted, 'ID')

        print(target_id, result)
        results.append(result)

    print('Mean score:', sum(results)/len(results), f'(n={len(results)})')

## Predict on test set

In [22]:
# NOTE: 並列化の意味は小さいかも
test_pred = generate_submission(test_seq, repo, seq_identity, n_structures=5, n_jobs=4)
print('submission rows:', len(test_pred))

test_pred.head()

submission rows: 9762


,ID,resname,resid,x_1,y_1,z_1,x_2,y_2,z_2,x_3,y_3,z_3,x_4,y_4,z_4,x_5,y_5,z_5
0,8ZNQ_1,A,1,12.133,-11.862,1.757,166.708,162.080,161.200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8ZNQ_2,C,2,NaN,NaN,NaN,167.160,157.024,159.502,193.176,143.370,155.162,NaN,NaN,NaN,9.981,-13.553,-4.518
2,8ZNQ_3,C,3,NaN,NaN,NaN,164.384,147.392,163.683,188.045,141.202,155.911,NaN,NaN,NaN,8.575,-8.310,5.003
3,8ZNQ_4,G,4,11.260,-7.167,5.547,160.163,145.020,166.617,NaN,NaN,NaN,-15.397,-2.768,4.068,12.275,-5.392,7.117
4,8ZNQ_5,U,5,12.239,-0.828,5.049,155.045,145.136,168.216,183.603,138.622,153.965,-6.594,-8.409,3.503,17.668,-2.604,8.109


## Submission

In [26]:
# Save submission
out_path = WORK_DIR / 'submission.csv'
test_pred.to_csv(out_path, index=False)
print('saved ->', out_path)

saved -> /Users/tatsuki/work/kaggle/kauto/competitions/rna2/data/output/submission.csv
